# Adaptive Shield Weekly Report (MVP)

"
"This notebook automates weekly reporting for **configuration drift** and **integration failure** alerts.

"
"## Pipeline
"
"1. Fetch alerts in lookback window
"
"2. Filter target alert types
"
"3. Enrich with security check details
"
"4. Enrich affected entities
"
"5. Build summary + entities tables
"
"6. Export XLSX + CSV

"
"ServiceNow enrichment is a stub in this MVP.

In [ ]:
# Cell 1: Standalone Initialization (functions + variables)
import os
import re
import json
import time
import logging
from collections import defaultdict, deque
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Callable, Deque, Iterable
from urllib.parse import parse_qs, urlparse

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import HTML, display


SUMMARY_COLUMNS = [
    "change_datetime",
    "security_check_name",
    "security_check_details",
    "remediation_steps",
    "impact_level",
    "current_status",
    "affected_entities_count",
    "affected_scope",
    "affected_entities_detail",
    "account_id",
    "account_name",
    "integration_id",
    "integration_name",
    "integration_alias",
    "security_check_id",
    "alert_id",
    "alert_type",
    "source",
    "source_id",
    "is_archived",
    "ticket_number",
    "ticket_owner",
    "ticket_status",
    "ticket_last_update_datetime",
    "ticket_last_update_content",
    "extracted_at_utc",
]

ENTITY_COLUMNS = [
    "account_id",
    "security_check_id",
    "alert_id",
    "entity_type",
    "entity_name",
    "entity_dismissed",
    "entity_dismissed_reason",
    "entity_dismiss_expiration_date",
    "entity_extra_context_json",
    "entity_usage_json",
    "entity_raw_json",
]

SNOW_COLUMNS = [
    "ticket_number",
    "ticket_owner",
    "ticket_status",
    "ticket_last_update_datetime",
    "ticket_last_update_content",
]

JOIN_KEY = "alert_id"
TARGET_ALERT_TYPES = {"configuration_drift", "integration_failure"}
_TYPE_ALIAS_MAP = {
    "security_check_degraded": "check_degraded",
    "security_check_degrade": "check_degraded",
    "securitycheckdegraded": "check_degraded",
    "threat": "threat",
}


class AdaptiveShieldClientError(RuntimeError):
    """Base exception for Adaptive Shield client failures."""


class AuthenticationError(AdaptiveShieldClientError):
    """Raised when API key is invalid or missing required permissions."""


class ApiRequestError(AdaptiveShieldClientError):
    """Raised when request retries are exhausted."""


class AdaptiveShieldClient:
    """Client for Adaptive Shield REST API."""

    def __init__(
        self,
        api_key: str,
        base_url: str = "https://api.adaptive-shield.com",
        *,
        rate_limit_per_minute: int = 90,
        timeout_seconds: int = 30,
        max_retries: int = 3,
        session: requests.Session | None = None,
        sleep_func: Callable[[float], None] = time.sleep,
        time_func: Callable[[], float] = time.monotonic,
    ) -> None:
        if not api_key:
            raise ValueError("AS_API_KEY is required")

        self.base_url = base_url.rstrip("/")
        self.timeout_seconds = timeout_seconds
        self.max_retries = max_retries
        self.rate_limit_per_minute = rate_limit_per_minute
        self._sleep = sleep_func
        self._time = time_func
        self._request_times: Deque[float] = deque()
        self._session = session or requests.Session()
        self._session.headers.update(
            {
                "Authorization": f"Token {api_key}",
                "Content-Type": "application/json",
            }
        )

    def get_accounts(self) -> list[dict[str, Any]]:
        records = self._paginate("GET", "/api/v1/accounts")
        return [item for item in records if isinstance(item, dict)]

    def get_alerts(
        self,
        account_id: str,
        from_date: str,
        to_date: str,
        alert_type: str | None = None,
    ) -> list[dict[str, Any]]:
        path = f"/api/v1/accounts/{account_id}/alerts"
        params: dict[str, Any] = {"from_date": from_date, "to_date": to_date}
        if alert_type:
            params["type"] = alert_type
        records = self._paginate("GET", path, params=params)
        return [item for item in records if isinstance(item, dict)]

    def get_security_check(self, account_id: str, security_check_id: str) -> dict[str, Any]:
        path = f"/api/v1/accounts/{account_id}/security_checks/{security_check_id}"
        payload = self._request("GET", path)
        if isinstance(payload, dict) and isinstance(payload.get("data"), dict):
            return payload["data"]
        if isinstance(payload, dict):
            return payload
        return {}

    def get_affected_entities(
        self,
        account_id: str,
        security_check_id: str,
        *,
        limit: int = 100,
        offset: int = 0,
    ) -> list[dict[str, Any]]:
        path = f"/api/v1/accounts/{account_id}/security_checks/{security_check_id}/affected"
        params: dict[str, Any] = {"limit": limit, "offset": offset}
        records = self._paginate("GET", path, params=params)
        return [item for item in records if isinstance(item, dict)]

    def get_integrations(self, account_id: str) -> list[dict[str, Any]]:
        path = f"/api/v1/accounts/{account_id}/integrations"
        records = self._paginate("GET", path)
        return [item for item in records if isinstance(item, dict)]

    def _paginate(
        self,
        method: str,
        path_or_url: str,
        params: dict[str, Any] | None = None,
    ) -> list[Any]:
        all_items: list[Any] = []
        seen_ids: set[str] = set()
        seen_pages: set[tuple[str, tuple[tuple[str, str], ...]]] = set()

        original_path = path_or_url
        next_path = path_or_url
        next_params = dict(params or {})

        while True:
            page_key = (
                next_path,
                tuple(sorted((str(k), str(v)) for k, v in (next_params or {}).items())),
            )
            if page_key in seen_pages:
                logging.warning("Pagination loop detected for %s", next_path)
                break
            seen_pages.add(page_key)

            payload = self._request(method, next_path, params=next_params or None)
            items, next_page_uri, pagination = self._extract_page(payload)

            for item in items:
                if isinstance(item, dict) and item.get("id") is not None:
                    item_id = str(item["id"])
                    if item_id in seen_ids:
                        continue
                    seen_ids.add(item_id)
                all_items.append(item)

            if next_page_uri:
                next_path = next_page_uri
                next_params = {}
                continue

            next_from_meta = self._next_from_meta(
                original_path=original_path,
                current_params=next_params,
                pagination=pagination,
            )
            if next_from_meta is None:
                break

            next_path, next_params = next_from_meta

        return all_items

    def _request(
        self,
        method: str,
        path_or_url: str,
        *,
        params: dict[str, Any] | None = None,
    ) -> Any:
        url = self._resolve_url(path_or_url)
        attempt = 0

        while True:
            self._throttle()
            try:
                response = self._session.request(
                    method=method.upper(),
                    url=url,
                    params=params,
                    timeout=self.timeout_seconds,
                )
            except requests.RequestException as exc:
                if attempt >= self.max_retries:
                    raise ApiRequestError(
                        f"Request failed after retries: {method} {url}"
                    ) from exc
                wait_seconds = float(2 ** attempt)
                logging.warning(
                    "Request error on %s %s, retrying in %.1fs: %s",
                    method,
                    url,
                    wait_seconds,
                    exc,
                )
                self._sleep(wait_seconds)
                attempt += 1
                continue

            if response.status_code == 401:
                raise AuthenticationError(
                    "Unauthorized (401). Verify AS_API_KEY and account access."
                )

            if response.status_code == 429 and attempt < self.max_retries:
                retry_after = self._parse_retry_after(response.headers.get("Retry-After"))
                wait_seconds = retry_after if retry_after is not None else float(2 ** attempt)
                logging.warning(
                    "Rate limited on %s %s, retrying in %.1fs",
                    method,
                    url,
                    wait_seconds,
                )
                self._sleep(wait_seconds)
                attempt += 1
                continue

            if 500 <= response.status_code < 600 and attempt < self.max_retries:
                wait_seconds = float(2 ** attempt)
                logging.warning(
                    "Server error %s on %s %s, retrying in %.1fs",
                    response.status_code,
                    method,
                    url,
                    wait_seconds,
                )
                self._sleep(wait_seconds)
                attempt += 1
                continue

            if response.status_code >= 400:
                snippet = (response.text or "").strip()[:500]
                raise ApiRequestError(
                    f"HTTP {response.status_code} for {method} {url}: {snippet}"
                )

            try:
                return response.json()
            except ValueError as exc:
                raise ApiRequestError(f"Invalid JSON response for {method} {url}") from exc

    def _throttle(self) -> None:
        now = self._time()
        while self._request_times and now - self._request_times[0] >= 60:
            self._request_times.popleft()

        if len(self._request_times) >= self.rate_limit_per_minute:
            wait_seconds = 60 - (now - self._request_times[0]) + 0.01
            logging.info("Throttling API calls for %.2fs", wait_seconds)
            self._sleep(wait_seconds)
            now = self._time()
            while self._request_times and now - self._request_times[0] >= 60:
                self._request_times.popleft()

        self._request_times.append(now)

    def _resolve_url(self, path_or_url: str) -> str:
        if path_or_url.startswith("http://") or path_or_url.startswith("https://"):
            return path_or_url
        return f"{self.base_url}/{path_or_url.lstrip('/')}"

    @staticmethod
    def _extract_page(payload: Any) -> tuple[list[Any], str | None, dict[str, Any] | None]:
        items: list[Any] = []
        next_page_uri: str | None = None
        pagination: dict[str, Any] | None = None

        if isinstance(payload, list):
            return payload, None, None

        if not isinstance(payload, dict):
            return [], None, None

        if isinstance(payload.get("data"), list):
            items = payload["data"]
        elif isinstance(payload.get("resources"), list):
            items = payload["resources"]
        elif isinstance(payload.get("data"), dict):
            data_obj = payload["data"]
            if isinstance(data_obj.get("result"), list):
                items = data_obj["result"]
        elif isinstance(payload.get("result"), list):
            items = payload["result"]

        if isinstance(payload.get("next_page_uri"), str):
            next_page_uri = payload["next_page_uri"]
        elif isinstance(payload.get("data"), dict) and isinstance(payload["data"].get("next_page_uri"), str):
            next_page_uri = payload["data"]["next_page_uri"]

        meta = payload.get("meta")
        if isinstance(meta, dict) and isinstance(meta.get("pagination"), dict):
            pagination = meta["pagination"]

        return items, next_page_uri, pagination

    @staticmethod
    def _parse_retry_after(header_value: str | None) -> float | None:
        if not header_value:
            return None
        try:
            return max(0.0, float(header_value))
        except (TypeError, ValueError):
            return None

    def _next_from_meta(
        self,
        *,
        original_path: str,
        current_params: dict[str, Any],
        pagination: dict[str, Any] | None,
    ) -> tuple[str, dict[str, Any]] | None:
        if not pagination:
            return None

        next_value = pagination.get("next")
        if isinstance(next_value, str):
            stripped = next_value.strip()
            if not stripped:
                return None
            if stripped.startswith("http://") or stripped.startswith("https://"):
                return stripped, {}
            if stripped.isdigit():
                params = dict(current_params)
                params["offset"] = int(stripped)
                return original_path, params
            parsed = urlparse(stripped)
            if parsed.query:
                params = dict(current_params)
                query_map = parse_qs(parsed.query)
                for key, values in query_map.items():
                    if values:
                        params[key] = values[-1]
                return (stripped if stripped.startswith("/") else original_path, params)

        if isinstance(next_value, (int, float)):
            params = dict(current_params)
            params["offset"] = int(next_value)
            return original_path, params

        if isinstance(next_value, dict):
            params = dict(current_params)
            for key in ("offset", "page", "cursor"):
                if key in next_value:
                    params[key] = next_value[key]
            return original_path, params

        if next_value in (None, "", False):
            offset = pagination.get("offset")
            limit = pagination.get("limit")
            total = pagination.get("total")
            if all(isinstance(v, int) for v in (offset, limit, total)):
                if offset + limit < total:
                    params = dict(current_params)
                    params["offset"] = offset + limit
                    params.setdefault("limit", limit)
                    return original_path, params

        return None


def normalize_alert_type(raw: Any) -> str:
    if raw is None:
        return ""

    value = str(raw).strip().lower()
    value = value.replace("-", "_").replace(" ", "_")
    value = re.sub(r"[^a-z0-9_]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return _TYPE_ALIAS_MAP.get(value, value)


def filter_target_alerts(
    rows: Iterable[dict[str, Any]],
    *,
    include_check_degraded: bool = False,
) -> list[dict[str, Any]]:
    targets = set(TARGET_ALERT_TYPES)
    if include_check_degraded:
        targets.add("check_degraded")

    filtered: list[dict[str, Any]] = []
    for row in rows:
        normalized = normalize_alert_type(row.get("alert_type") or row.get("type"))
        if bool(row.get("is_archived")):
            continue
        if normalized not in targets:
            continue

        item = dict(row)
        item["alert_type_normalized"] = normalized
        filtered.append(item)

    return filtered


def extract_security_check_id(alert: dict[str, Any]) -> str:
    for key in ("source_id", "security_check_id"):
        value = alert.get(key)
        if value:
            return str(value)

    api_link = alert.get("security_check_api_link")
    if not api_link:
        return ""

    match = re.search(r"/security_checks/([^/?#]+)", str(api_link))
    if match:
        return match.group(1)
    return ""


def _to_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"), default=str)


def _fallback_status_from_alert(alert: dict[str, Any]) -> str:
    normalized = normalize_alert_type(alert.get("alert_type") or alert.get("type"))
    if normalized == "integration_failure":
        return "Failed"
    if normalized == "configuration_drift":
        return "Drifted"
    if normalized == "check_degraded":
        return "Degraded"
    return ""


def build_entities_df(entity_records: Iterable[dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    for record in entity_records:
        account_id = record.get("account_id")
        alert_id = record.get("alert_id")
        security_check_id = record.get("security_check_id")
        entity = record.get("entity") if isinstance(record.get("entity"), dict) else record
        if not isinstance(entity, dict):
            continue

        rows.append(
            {
                "account_id": account_id,
                "security_check_id": security_check_id,
                "alert_id": alert_id,
                "entity_type": entity.get("type"),
                "entity_name": entity.get("entity_name") or entity.get("name"),
                "entity_dismissed": entity.get("dismissed"),
                "entity_dismissed_reason": entity.get("dismissed_reason"),
                "entity_dismiss_expiration_date": entity.get("dismiss_expiration_date"),
                "entity_extra_context_json": _to_json(entity.get("extra_context")),
                "entity_usage_json": _to_json(entity.get("usage")),
                "entity_raw_json": _to_json(entity),
            }
        )

    return pd.DataFrame(rows, columns=ENTITY_COLUMNS)


def build_summary_df(
    *,
    alerts: Iterable[dict[str, Any]],
    checks: Iterable[dict[str, Any]],
    entities: Iterable[dict[str, Any]],
    accounts: Iterable[dict[str, Any]] | None = None,
    extracted_at_utc: str | None = None,
) -> pd.DataFrame:
    extracted_at = extracted_at_utc or datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    account_name_by_id: dict[str, str] = {}
    if accounts:
        for account in accounts:
            account_id = str(account.get("id") or account.get("account_id") or "")
            if account_id:
                account_name_by_id[account_id] = str(account.get("name") or account.get("account_name") or "")

    checks_by_alert: dict[str, dict[str, Any]] = {}
    checks_by_pair: dict[tuple[str, str], dict[str, Any]] = {}
    for check in checks:
        if not isinstance(check, dict):
            continue
        alert_id = str(check.get("alert_id") or "")
        account_id = str(check.get("account_id") or "")
        security_check_id = str(
            check.get("security_check_id")
            or check.get("id")
            or check.get("source_id")
            or ""
        )

        if alert_id:
            checks_by_alert[alert_id] = check
        if account_id and security_check_id:
            checks_by_pair[(account_id, security_check_id)] = check

    entity_names_by_alert: dict[str, list[str]] = defaultdict(list)
    entity_count_by_alert: dict[str, int] = defaultdict(int)
    for entity in entities:
        if not isinstance(entity, dict):
            continue
        alert_id = str(entity.get("alert_id") or "")
        if not alert_id:
            continue
        entity_name = entity.get("entity_name")
        if entity_name:
            entity_names_by_alert[alert_id].append(str(entity_name))
        entity_count_by_alert[alert_id] += 1

    rows: list[dict[str, Any]] = []
    for alert in alerts:
        if not isinstance(alert, dict):
            continue

        alert_id = str(alert.get("id") or alert.get("alert_id") or "")
        account_id = str(alert.get("account_id") or "")
        source_id = str(alert.get("source_id") or "")
        security_check_id = extract_security_check_id(alert) or source_id
        check = checks_by_alert.get(alert_id)
        if check is None and account_id and security_check_id:
            check = checks_by_pair.get((account_id, security_check_id), {})
        check = check or {}

        integration_obj = alert.get("integration") if isinstance(alert.get("integration"), dict) else {}
        if not integration_obj and isinstance(check.get("integration"), dict):
            integration_obj = check["integration"]

        is_global = bool(check.get("is_global"))
        entity_names = sorted(set(entity_names_by_alert.get(alert_id, [])))
        affected_diff = alert.get("affected_diff")
        affected_diff_names: list[str] = []
        if isinstance(affected_diff, list):
            affected_diff_names = [str(item) for item in affected_diff if item is not None]

        affected_scope = "global" if is_global else "unresolved"
        if not is_global:
            if entity_names:
                affected_scope = "entity"
            elif affected_diff_names:
                affected_scope = "entity_diff"

        if is_global:
            affected_entities_count: Any = "global"
        else:
            affected_entities_count = (
                check.get("affected")
                if check.get("affected") not in (None, "")
                else alert.get("new_affected_count")
            )
            if affected_entities_count in (None, "") and entity_count_by_alert.get(alert_id):
                affected_entities_count = entity_count_by_alert[alert_id]

        if is_global:
            affected_entities_detail = "global"
        elif entity_names:
            affected_entities_detail = "; ".join(entity_names)
        elif affected_diff_names:
            affected_entities_detail = "; ".join(affected_diff_names)
        else:
            affected_entities_detail = ""

        current_status = check.get("status") or _fallback_status_from_alert(alert)

        rows.append(
            {
                "change_datetime": alert.get("timestamp"),
                "security_check_name": check.get("name") or check.get("title"),
                "security_check_details": check.get("details") or alert.get("description"),
                "remediation_steps": check.get("remediation_plan"),
                "impact_level": check.get("impact"),
                "current_status": current_status,
                "affected_entities_count": affected_entities_count,
                "affected_scope": affected_scope,
                "affected_entities_detail": affected_entities_detail,
                "account_id": account_id,
                "account_name": account_name_by_id.get(account_id, ""),
                "integration_id": integration_obj.get("id") or check.get("integration_id") or check.get("Integration_id"),
                "integration_name": integration_obj.get("name") or check.get("integration_name") or check.get("saas_name"),
                "integration_alias": integration_obj.get("alias") or check.get("integration_alias"),
                "security_check_id": security_check_id,
                "alert_id": alert_id,
                "alert_type": normalize_alert_type(alert.get("alert_type") or alert.get("type")),
                "source": alert.get("source"),
                "source_id": source_id,
                "is_archived": bool(alert.get("is_archived")),
                "ticket_number": None,
                "ticket_owner": None,
                "ticket_status": None,
                "ticket_last_update_datetime": None,
                "ticket_last_update_content": None,
                "extracted_at_utc": extracted_at,
            }
        )

    return pd.DataFrame(rows, columns=SUMMARY_COLUMNS)


def fetch_related_tickets(summary_df: pd.DataFrame, lookback_days: int) -> pd.DataFrame:
    _ = summary_df
    _ = lookback_days
    return pd.DataFrame(columns=[JOIN_KEY, *SNOW_COLUMNS])


def merge_snow_columns(summary_df: pd.DataFrame, snow_df: pd.DataFrame | None) -> pd.DataFrame:
    result = summary_df.copy()
    for column in SNOW_COLUMNS:
        if column not in result.columns:
            result[column] = None

    if snow_df is None or snow_df.empty:
        return result
    if JOIN_KEY not in snow_df.columns:
        return result

    selected_columns = [JOIN_KEY] + [col for col in SNOW_COLUMNS if col in snow_df.columns]
    merged = result.drop(columns=SNOW_COLUMNS, errors="ignore").merge(
        snow_df[selected_columns].drop_duplicates(subset=[JOIN_KEY], keep="last"),
        on=JOIN_KEY,
        how="left",
    )

    for column in SNOW_COLUMNS:
        if column not in merged.columns:
            merged[column] = None

    return merged


def export_all(
    summary_df: pd.DataFrame | None,
    entities_df: pd.DataFrame | None,
    errors_df: pd.DataFrame | None,
    output_dir: str,
    ts: str,
    *,
    export_xlsx: bool = True,
    export_csv: bool = True,
) -> dict[str, str]:
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    summary_df = summary_df if summary_df is not None else pd.DataFrame()
    entities_df = entities_df if entities_df is not None else pd.DataFrame()
    errors_df = errors_df if errors_df is not None else pd.DataFrame()

    results: dict[str, Any] = {
        "xlsx_path": "",
        "summary_csv_path": "",
        "entities_csv_path": "",
        "errors_csv_path": "",
    }

    if export_xlsx:
        xlsx_path = output_path / f"AS_Weekly_Report_{ts}.xlsx"
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
            summary_df.to_excel(writer, sheet_name="summary", index=False)
            entities_df.to_excel(writer, sheet_name="entities", index=False)
            errors_df.to_excel(writer, sheet_name="errors", index=False)
        results["xlsx_path"] = str(xlsx_path)

    if export_csv:
        summary_csv_path = output_path / f"AS_Weekly_Summary_{ts}.csv"
        entities_csv_path = output_path / f"AS_Weekly_Entities_{ts}.csv"
        errors_csv_path = output_path / f"AS_Weekly_Errors_{ts}.csv"

        summary_df.to_csv(summary_csv_path, index=False)
        entities_df.to_csv(entities_csv_path, index=False)
        errors_df.to_csv(errors_csv_path, index=False)

        results["summary_csv_path"] = str(summary_csv_path)
        results["entities_csv_path"] = str(entities_csv_path)
        results["errors_csv_path"] = str(errors_csv_path)

    return {k: str(v) for k, v in results.items()}



STATUS_ORDER = ["failed", "degraded", "drifted", "pending", "dismissed", "passed", "unknown"]


def normalize_status(raw: Any) -> str:
    if raw is None:
        return "unknown"
    value = str(raw).strip().lower()
    if not value:
        return "unknown"
    if "fail" in value:
        return "failed"
    if "pass" in value:
        return "passed"
    if "degrad" in value:
        return "degraded"
    if "drift" in value:
        return "drifted"
    if "dismiss" in value:
        return "dismissed"
    if "pend" in value:
        return "pending"
    return value


def _parse_ts(ts: Any) -> datetime:
    if ts is None:
        return datetime.min.replace(tzinfo=timezone.utc)
    raw = str(ts).strip()
    if not raw:
        return datetime.min.replace(tzinfo=timezone.utc)
    try:
        return datetime.fromisoformat(raw.replace("Z", "+00:00"))
    except ValueError:
        return datetime.min.replace(tzinfo=timezone.utc)


def _escape(value: Any) -> str:
    import html
    return html.escape(str(value if value is not None else ""))


def _style_badge(text: str, kind: str) -> str:
    return f'<span class="as-badge as-badge-{kind}">{_escape(text)}</span>'


def _status_rank(status: str) -> int:
    try:
        return STATUS_ORDER.index(status)
    except ValueError:
        return len(STATUS_ORDER)


def _detect_flip_flop(statuses: list[str]) -> bool:
    clean = [normalize_status(s) for s in statuses if s is not None]
    if len(set(clean)) < 2:
        return False
    has_failed = any(s == "failed" for s in clean)
    has_passed = any(s == "passed" for s in clean)
    return has_failed and has_passed


def build_drift_groups(summary_df: pd.DataFrame, entities_df: pd.DataFrame) -> list[dict[str, Any]]:
    if summary_df.empty:
        return []

    groups: dict[tuple[str, str, str], list[dict[str, Any]]] = defaultdict(list)
    for row in summary_df.to_dict("records"):
        account_id = str(row.get("account_id") or "")
        integration_key = (
            str(row.get("integration_id") or "")
            or str(row.get("integration_alias") or "")
            or str(row.get("integration_name") or "")
            or "unknown_integration"
        )
        security_key = str(row.get("security_check_id") or row.get("security_check_name") or "unknown_check")
        groups[(account_id, integration_key, security_key)].append(row)

    entity_map: dict[str, list[str]] = defaultdict(list)
    if not entities_df.empty:
        for entity in entities_df.to_dict("records"):
            alert_id = str(entity.get("alert_id") or "")
            name = entity.get("entity_name")
            if alert_id and name:
                entity_map[alert_id].append(str(name))

    result: list[dict[str, Any]] = []
    for (_, _, _), rows in groups.items():
        rows_sorted = sorted(rows, key=lambda r: _parse_ts(r.get("change_datetime")), reverse=True)
        current = rows_sorted[0]
        current_status = normalize_status(current.get("current_status"))
        statuses = [normalize_status(r.get("current_status")) for r in rows_sorted]
        flip_flop = _detect_flip_flop(statuses)

        alert_ids = [str(r.get("alert_id") or "") for r in rows_sorted if r.get("alert_id")]
        entity_names: list[str] = []
        for alert_id in alert_ids:
            entity_names.extend(entity_map.get(alert_id, []))
        entity_names = sorted(set(entity_names))

        if not entity_names:
            fallback = str(current.get("affected_entities_detail") or "").strip()
            if fallback and fallback.lower() != "global":
                entity_names = [v.strip() for v in fallback.split(";") if v.strip()]

        affected_scope = str(current.get("affected_scope") or "").lower()
        if affected_scope == "global":
            affected_count = "global"
        else:
            affected_count = current.get("affected_entities_count")
            if affected_count in (None, "") and entity_names:
                affected_count = len(entity_names)

        result.append(
            {
                "account_id": current.get("account_id"),
                "account_name": current.get("account_name"),
                "integration_id": current.get("integration_id"),
                "integration_name": current.get("integration_name"),
                "integration_alias": current.get("integration_alias"),
                "security_check_id": current.get("security_check_id"),
                "security_check_name": current.get("security_check_name"),
                "security_check_details": current.get("security_check_details"),
                "remediation_steps": current.get("remediation_steps"),
                "impact_level": current.get("impact_level"),
                "current_status": current_status,
                "change_datetime": current.get("change_datetime"),
                "affected_scope": affected_scope,
                "affected_count": affected_count,
                "entities": entity_names,
                "flip_flop": flip_flop,
                "history": rows_sorted,
            }
        )

    result.sort(key=lambda g: (_status_rank(g["current_status"]), _parse_ts(g["change_datetime"]), str(g.get("integration_alias") or g.get("integration_name") or "")), reverse=False)
    return result


def render_configuration_drifts_ui(summary_df: pd.DataFrame, entities_df: pd.DataFrame) -> None:
    groups = build_drift_groups(summary_df, entities_df)
    if not groups:
        display(HTML('<div style="padding:10px;border:1px solid #ddd;border-radius:8px;">No configuration drift records found for the current window.</div>'))
        return

    grouped_by_status: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for group in groups:
        grouped_by_status[group["current_status"]].append(group)

    status_list = sorted(grouped_by_status.keys(), key=_status_rank)

    html_parts = [
        '<style>',
        '.as-ui {font-family: Arial, sans-serif; font-size: 13px; line-height: 1.4;}',
        '.as-title {font-size: 18px; font-weight: 700; margin-bottom: 10px;}',
        '.as-meta {color: #666; margin-bottom: 12px;}',
        '.as-status-group {border: 1px solid #ddd; border-radius: 10px; margin-bottom: 10px; background: #fff;}',
        '.as-status-summary {cursor: pointer; font-weight: 700; padding: 10px 12px; background: #f8f9fa; border-radius: 10px;}',
        '.as-group-body {padding: 10px 12px;}',
        '.as-card {border: 1px solid #e5e7eb; border-radius: 8px; margin-bottom: 10px; background: #fff;}',
        '.as-card.as-flipflop {border-left: 5px solid #f59e0b;}',
        '.as-card-summary {cursor: pointer; padding: 8px 10px; font-weight: 600; background: #fdfdfd;}',
        '.as-card-content {padding: 8px 10px 10px 10px;}',
        '.as-line {margin-bottom: 6px;}',
        '.as-key {font-weight: 600; color: #111827;}',
        '.as-val {color: #374151;}',
        '.as-badge {display: inline-block; margin-left: 6px; padding: 2px 6px; border-radius: 999px; font-size: 11px;}',
        '.as-badge-status-failed {background: #fee2e2; color: #991b1b;}',
        '.as-badge-status-passed {background: #dcfce7; color: #166534;}',
        '.as-badge-status-other {background: #e5e7eb; color: #374151;}',
        '.as-badge-flip {background: #fef3c7; color: #92400e;}',
        '.as-history table {width: 100%; border-collapse: collapse;}',
        '.as-history th, .as-history td {border: 1px solid #e5e7eb; padding: 4px 6px; text-align: left;}',
        '.as-history th {background: #f9fafb;}',
        '.as-entities {margin-top: 6px;}',
        '.as-entities ul {margin: 6px 0 0 16px;}',
        '.as-entities li {margin-bottom: 2px;}',
        '</style>',
        '<div class="as-ui">',
        '<div class="as-title">Configuration Drift Review</div>',
        f'<div class="as-meta">Grouped checks: {len(groups)}. Failed groups are shown first. Passed groups are folded by default.</div>',
    ]

    for status in status_list:
        items = grouped_by_status[status]
        is_passed = status == "passed"
        open_attr = '' if is_passed else ' open'
        status_label = status.capitalize()
        html_parts.append(f'<details class="as-status-group"{open_attr}>')
        html_parts.append(f'<summary class="as-status-summary">{_escape(status_label)} ({len(items)})</summary>')
        html_parts.append('<div class="as-group-body">')

        for item in items:
            integration_display = item.get("integration_alias") or item.get("integration_name") or item.get("integration_id") or "Unknown integration"
            check_display = item.get("security_check_name") or item.get("security_check_id") or "Unknown check"
            status_kind = "failed" if item["current_status"] == "failed" else ("passed" if item["current_status"] == "passed" else "other")
            status_badge = _style_badge(item["current_status"].capitalize(), f"status-{status_kind}")
            flip_badge = _style_badge("FLIP-FLOP", "flip") if item["flip_flop"] else ""
            card_class = "as-card as-flipflop" if item["flip_flop"] else "as-card"

            entity_count = item["affected_count"] if item["affected_count"] not in (None, "") else len(item["entities"])
            if entity_count in (None, ""):
                entity_count = 0

            html_parts.append(f'<details class="{card_class}" open>')
            html_parts.append(
                '<summary class="as-card-summary">'
                f'{_escape(integration_display)} | {_escape(check_display)} '
                f'{status_badge}{flip_badge} '
                f'<span style="color:#6b7280;font-weight:500;">(affected: {_escape(entity_count)})</span>'
                '</summary>'
            )
            html_parts.append('<div class="as-card-content">')
            html_parts.append(f'<div class="as-line"><span class="as-key">Last change:</span> <span class="as-val">{_escape(item.get("change_datetime") or "")}</span></div>')
            html_parts.append(f'<div class="as-line"><span class="as-key">Impact:</span> <span class="as-val">{_escape(item.get("impact_level") or "")}</span></div>')
            html_parts.append(f'<div class="as-line"><span class="as-key">Details:</span> <span class="as-val">{_escape(item.get("security_check_details") or "")}</span></div>')
            html_parts.append(f'<div class="as-line"><span class="as-key">Remediation:</span> <span class="as-val">{_escape(item.get("remediation_steps") or "")}</span></div>')

            html_parts.append('<details class="as-entities">')
            html_parts.append(f'<summary>Affected entities ({_escape(entity_count)})</summary>')
            if str(item.get("affected_scope") or "").lower() == "global":
                html_parts.append('<div style="margin-top:6px;">Global setting (not entity-scoped).</div>')
            elif item["entities"]:
                html_parts.append('<ul>')
                for entity_name in item["entities"]:
                    html_parts.append(f'<li>{_escape(entity_name)}</li>')
                html_parts.append('</ul>')
            else:
                html_parts.append('<div style="margin-top:6px;">No entity list returned.</div>')
            html_parts.append('</details>')

            html_parts.append('<details class="as-history" style="margin-top:8px;">')
            html_parts.append(f'<summary>History ({len(item["history"])} events)</summary>')
            html_parts.append('<table><thead><tr><th>Datetime</th><th>Status</th><th>Alert Type</th><th>Affected</th></tr></thead><tbody>')
            for row in item["history"]:
                row_status = normalize_status(row.get("current_status")).capitalize()
                row_type = normalize_alert_type(row.get("alert_type"))
                row_affected = row.get("affected_entities_count")
                html_parts.append(
                    '<tr>'
                    f'<td>{_escape(row.get("change_datetime") or "")}</td>'
                    f'<td>{_escape(row_status)}</td>'
                    f'<td>{_escape(row_type)}</td>'
                    f'<td>{_escape(row_affected)}</td>'
                    '</tr>'
                )
            html_parts.append('</tbody></table>')
            html_parts.append('</details>')

            html_parts.append('</div>')
            html_parts.append('</details>')

        html_parts.append('</div>')
        html_parts.append('</details>')

    html_parts.append('</div>')
    display(HTML(''.join(html_parts)))

def env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / ".env.example").exists() and (candidate / "requirements.txt").exists():
            return candidate
    return cwd


ROOT_DIR = resolve_project_root()
load_dotenv(ROOT_DIR / ".env")

RUN_TS = datetime.now(timezone.utc)
LOOKBACK_DAYS = int(os.getenv("LOOKBACK_DAYS", "3"))
FROM_DATE = (RUN_TS - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%dT%H:%M:%SZ")
TO_DATE = RUN_TS.strftime("%Y-%m-%dT%H:%M:%SZ")

config = {
    "as_api_key": os.getenv("AS_API_KEY", "").strip(),
    "as_base_url": os.getenv("AS_BASE_URL", "https://api.adaptive-shield.com").strip(),
    "as_account_ids": [
        account.strip()
        for account in os.getenv("AS_ACCOUNT_IDS", "").split(",")
        if account.strip()
    ],
    "lookback_days": LOOKBACK_DAYS,
    "from_date": FROM_DATE,
    "to_date": TO_DATE,
    "output_root": os.getenv("OUTPUT_ROOT", "output").strip(),
    "export_xlsx": env_bool("EXPORT_XLSX", True),
    "export_csv": env_bool("EXPORT_CSV", True),
    "snow_enabled": env_bool("SNOW_ENABLED", False),
    "rate_limit_per_minute": int(os.getenv("RATE_LIMIT_PER_MINUTE", "90")),
    "request_timeout_seconds": int(os.getenv("REQUEST_TIMEOUT_SECONDS", "30")),
    "max_retries": int(os.getenv("MAX_RETRIES", "3")),
}

if not config["as_api_key"]:
    raise ValueError("AS_API_KEY is required. Add it to .env before running.")

RUN_DAY_DIR = ROOT_DIR / config["output_root"] / RUN_TS.strftime("%Y-%m-%d")
RUN_DAY_DIR.mkdir(parents=True, exist_ok=True)

pipeline_errors = []

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
print("Configuration loaded:")
print({k: v for k, v in config.items() if k != "as_api_key"})
print("Project root:", ROOT_DIR)
print("Output directory:", RUN_DAY_DIR)


In [ ]:
# Cell 2: API Client Initialization

client = AdaptiveShieldClient(
    api_key=config["as_api_key"],
    base_url=config["as_base_url"],
    rate_limit_per_minute=config["rate_limit_per_minute"],
    timeout_seconds=config["request_timeout_seconds"],
    max_retries=config["max_retries"],
)

print("AdaptiveShieldClient initialized")

In [ ]:
# Cell 3: Get Accounts
accounts_records = []

if config["as_account_ids"]:
    for account_id in config["as_account_ids"]:
        accounts_records.append({"id": account_id, "name": ""})
else:
    accounts_records = client.get_accounts()

accounts_df = pd.DataFrame(accounts_records)
if not accounts_df.empty:
    accounts_df = accounts_df.rename(columns={"id": "account_id", "name": "account_name"})
    for column in ["account_id", "account_name"]:
        if column not in accounts_df.columns:
            accounts_df[column] = ""
    accounts_df = accounts_df[["account_id", "account_name"]]
else:
    accounts_df = pd.DataFrame(columns=["account_id", "account_name"])

display(accounts_df.head(20))
print("Accounts found:", len(accounts_df))


In [ ]:
# Cell 4: Fetch Alerts

TARGET_ALERT_TYPES = ["configuration_drift", "integration_failure"]

alerts_raw_rows = []
for account_id in accounts_df["account_id"].dropna().astype(str).tolist():
    for alert_type in TARGET_ALERT_TYPES:
        try:
            rows = client.get_alerts(
                account_id=account_id,
                from_date=config["from_date"],
                to_date=config["to_date"],
                alert_type=alert_type,
            )
            for row in rows:
                row.setdefault("account_id", account_id)
            alerts_raw_rows.extend(rows)
        except Exception as exc:
            pipeline_errors.append(
                {
                    "stage": "alerts",
                    "account_id": account_id,
                    "security_check_id": None,
                    "alert_id": None,
                    "message": str(exc),
                }
            )

alerts_raw_df = pd.DataFrame(alerts_raw_rows)
if not alerts_raw_df.empty and "id" in alerts_raw_df.columns:
    alerts_raw_df = alerts_raw_df.drop_duplicates(subset=["account_id", "id"], keep="first")

alerts_filtered_rows = filter_target_alerts(alerts_raw_df.to_dict("records"), include_check_degraded=False)
alerts_filtered_df = pd.DataFrame(alerts_filtered_rows)

display(alerts_filtered_df.head(20))
print("Raw alerts:", len(alerts_raw_df), "| Filtered alerts:", len(alerts_filtered_df))

In [ ]:
# Cell 5: Enrich Security Checks

check_cache = {}
check_rows = []

for alert in alerts_filtered_df.to_dict("records"):
    alert_id = str(alert.get("id") or "")
    account_id = str(alert.get("account_id") or "")
    security_check_id = extract_security_check_id(alert)

    if not account_id or not security_check_id:
        pipeline_errors.append(
            {
                "stage": "security_check",
                "account_id": account_id,
                "security_check_id": security_check_id,
                "alert_id": alert_id,
                "message": "Missing account_id or security_check_id",
            }
        )
        check_rows.append(
            {
                "alert_id": alert_id,
                "account_id": account_id,
                "security_check_id": security_check_id,
            }
        )
        continue

    cache_key = (account_id, security_check_id)
    if cache_key not in check_cache:
        try:
            check_cache[cache_key] = client.get_security_check(account_id, security_check_id)
        except Exception as exc:
            check_cache[cache_key] = {}
            pipeline_errors.append(
                {
                    "stage": "security_check",
                    "account_id": account_id,
                    "security_check_id": security_check_id,
                    "alert_id": alert_id,
                    "message": str(exc),
                }
            )

    check_data = dict(check_cache[cache_key])
    check_rows.append(
        {
            "alert_id": alert_id,
            "account_id": account_id,
            "security_check_id": security_check_id,
            **check_data,
        }
    )

checks_df = pd.DataFrame(check_rows)
display(checks_df.head(20))
print("Security checks enriched:", len(checks_df))

In [ ]:
# Cell 6: Enrich Affected Entities

entity_records = []
affected_resolve_log_rows = []
affected_cache = {}

for alert in alerts_filtered_df.to_dict("records"):
    alert_id = str(alert.get("id") or "")
    account_id = str(alert.get("account_id") or "")
    security_check_id = extract_security_check_id(alert)

    if "alert_id" in checks_df.columns:
        check_row = checks_df[checks_df["alert_id"].astype(str) == alert_id]
        check_obj = check_row.iloc[0].to_dict() if not check_row.empty else {}
    else:
        check_obj = {}
    is_global = bool(check_obj.get("is_global"))

    if is_global:
        affected_resolve_log_rows.append(
            {
                "alert_id": alert_id,
                "account_id": account_id,
                "security_check_id": security_check_id,
                "resolve_status": "global",
                "message": "Security check marked as global",
            }
        )
        continue

    cache_key = (account_id, security_check_id)
    entities = []
    if cache_key in affected_cache:
        entities = affected_cache[cache_key]
    else:
        try:
            entities = client.get_affected_entities(account_id, security_check_id)
            affected_cache[cache_key] = entities
        except Exception as exc:
            pipeline_errors.append(
                {
                    "stage": "affected_entities",
                    "account_id": account_id,
                    "security_check_id": security_check_id,
                    "alert_id": alert_id,
                    "message": str(exc),
                }
            )
            affected_resolve_log_rows.append(
                {
                    "alert_id": alert_id,
                    "account_id": account_id,
                    "security_check_id": security_check_id,
                    "resolve_status": "unresolved",
                    "message": "Affected API failed",
                }
            )
            continue

    for entity in entities:
        entity_records.append(
            {
                "alert_id": alert_id,
                "account_id": account_id,
                "security_check_id": security_check_id,
                "entity": entity,
            }
        )

    affected_resolve_log_rows.append(
        {
            "alert_id": alert_id,
            "account_id": account_id,
            "security_check_id": security_check_id,
            "resolve_status": "fetched",
            "message": f"Fetched {len(entities)} entities",
        }
    )

entities_df = build_entities_df(entity_records)
affected_resolve_log_df = pd.DataFrame(affected_resolve_log_rows)

display(entities_df.head(20))
display(affected_resolve_log_df.head(20))
print("Entity rows:", len(entities_df))

In [ ]:
# Cell 7: Build Summary Table

summary_df = build_summary_df(
    alerts=alerts_filtered_df.to_dict("records"),
    checks=checks_df.to_dict("records"),
    entities=entities_df.to_dict("records"),
    accounts=accounts_df.rename(columns={"account_id": "id", "account_name": "name"}).to_dict("records"),
    extracted_at_utc=RUN_TS.strftime("%Y-%m-%dT%H:%M:%SZ"),
)

display(summary_df.head(20))
print("Summary rows:", len(summary_df))

In [ ]:
# Cell 8: Configuration Drift UI
if "alert_type" in summary_df.columns:
    config_drift_df = summary_df[
        summary_df["alert_type"].fillna("").astype(str).str.lower() == "configuration_drift"
    ].copy()
else:
    config_drift_df = summary_df.copy()

render_configuration_drifts_ui(config_drift_df, entities_df)


In [ ]:
# Cell 9: ServiceNow Stub

snow_df = fetch_related_tickets(summary_df, config["lookback_days"]) if config["snow_enabled"] else fetch_related_tickets(summary_df, config["lookback_days"])
summary_with_snow_df = merge_snow_columns(summary_df, snow_df)

display(snow_df.head(20))
display(summary_with_snow_df.head(20))

In [ ]:
# Cell 10: Data Quality Checks

quality_rows = []

missing_summary_columns = [col for col in SUMMARY_COLUMNS if col not in summary_with_snow_df.columns]
missing_entity_columns = [col for col in ENTITY_COLUMNS if col not in entities_df.columns]

quality_rows.append(
    {
        "check": "missing_summary_columns",
        "value": len(missing_summary_columns),
        "details": ", ".join(missing_summary_columns),
    }
)
quality_rows.append(
    {
        "check": "missing_entity_columns",
        "value": len(missing_entity_columns),
        "details": ", ".join(missing_entity_columns),
    }
)

if not summary_with_snow_df.empty:
    duplicate_alert_keys = summary_with_snow_df.duplicated(subset=["account_id", "alert_id"]).sum()
else:
    duplicate_alert_keys = 0

quality_rows.append(
    {
        "check": "duplicate_account_alert_keys",
        "value": int(duplicate_alert_keys),
        "details": "subset=[account_id, alert_id]",
    }
)

for column in ["change_datetime", "security_check_name", "impact_level", "current_status"]:
    if column in summary_with_snow_df.columns and not summary_with_snow_df.empty:
        null_ratio = float(summary_with_snow_df[column].isna().mean())
    else:
        null_ratio = 1.0
    quality_rows.append(
        {
            "check": f"null_ratio::{column}",
            "value": round(null_ratio, 4),
            "details": "fraction of null values",
        }
    )

quality_report_df = pd.DataFrame(quality_rows)
errors_df = pd.DataFrame(pipeline_errors)

display(quality_report_df)
display(errors_df.head(20))
print("Pipeline errors:", len(errors_df))

In [ ]:
# Cell 11: Export Files

ts = RUN_TS.strftime("%Y%m%d_%H%M%S")
export_paths = export_all(
    summary_df=summary_with_snow_df,
    entities_df=entities_df,
    errors_df=errors_df,
    output_dir=str(RUN_DAY_DIR),
    ts=ts,
    export_xlsx=config["export_xlsx"],
    export_csv=config["export_csv"],
)

print("Export paths:")
for key, value in export_paths.items():
    print(f"- {key}: {value}")